<a href="https://colab.research.google.com/github/repulsivityy/learning-LLMs/blob/main/notebooks/00_mechanics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Initial Setup

Setting up the colab

In [ ]:
import os

REPO_URL = "https://github.com/repulsivityy/learning-LLMs.git"
REPO_DIR = "/content/learning-LLMs"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git remote set-url origin https://{token}@github.com/repulsivityy/learning-LLMs.git

In [ ]:
!pip install -q torch numpy
!mkdir -p notebooks src

In [ ]:
%%writefile src/tokenizer.py
from collections import defaultdict


def get_pair_counts(word_freqs):
    pair_counts = defaultdict(int)
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pair_counts[pair] += freq
    return pair_counts


def merge_pair(pair, word_freqs):
    new_word_freqs = {}
    for word, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_word_freqs[tuple(new_word)] = freq
    return new_word_freqs


def train_bpe(word_freqs, num_merges):
    word_freqs = dict(word_freqs)
    merges = []
    for step in range(num_merges):
        pair_counts = get_pair_counts(word_freqs)
        if not pair_counts:
            break
        best_pair = max(pair_counts, key=pair_counts.get)
        word_freqs = merge_pair(best_pair, word_freqs)
        merges.append(best_pair)
        print(f"Merge {step + 1}: {best_pair}  (count={pair_counts[best_pair]})")
    return word_freqs, merges

## Module 0 — Tokenization: Byte-Pair Encoding (BPE)

**Why it matters:** a language model never sees text — only a sequence of integers.
Tokenization is the text ↔ integer conversion. BPE is the algorithm nearly every
modern LLM uses to decide what those "chunks" should be.

**ELI5:** imagine a box of individual letter tiles. Every time two tiles keep
turning up next to each other ("t" + "h" in "the", "this", "that"), you glue
them into one bigger tile. Do that thousands of times and your box ends up
with tiles for whole common words, plus loose letters for anything unusual —
so you're never stuck on a word you've never seen.

In [ ]:
import sys
sys.path.append('/content/learning-LLMs/src')

from tokenizer import get_pair_counts, merge_pair, train_bpe

### Toy corpus

The classic example from the original BPE paper (Sennrich et al., 2016) —
useful because we know exactly what the "correct" output should be, so we
can check our implementation against it.

Each word is a tuple of characters plus an end-of-word marker `</w>`, mapped
to how many times it appears in the corpus.

In [ ]:
word_freqs = {
    ('l', 'o', 'w', '</w>'): 5,
    ('l', 'o', 'w', 'e', 'r', '</w>'): 2,
    ('n', 'e', 'w', 'e', 's', 't', '</w>'): 6,
    ('w', 'i', 'd', 'e', 's', 't', '</w>'): 3,
}
word_freqs

### Step 1 — `get_pair_counts`

Counts every adjacent symbol pair across the corpus, weighted by how often
each word appears.

**ELI5:** for every word, look at every two touching letters, and add "how
many times this whole word shows up" to that pair's running total.

In [ ]:
pair_counts = get_pair_counts(word_freqs)
sorted(pair_counts.items(), key=lambda x: -x[1])[:5]

### Step 2 — `merge_pair`

Fuses the winning pair everywhere it occurs.

**ELI5:** walk through each word letter by letter; every time you spot the
exact pair you're gluing, weld those two into one tile and hop over both.

In [ ]:
merged = merge_pair(('e', 's'), word_freqs)
merged

### Step 3 — `train_bpe`: the full training loop

Repeats "find the most frequent pair → merge it" for a fixed number of
steps. The **ordered list of merges is the entire trained tokenizer** — to
tokenize new text later, you replay these same merges in this same order.

In [ ]:
final_word_freqs, merges = train_bpe(word_freqs, num_merges=8)
merges

In [ ]:
final_word_freqs

### Step 4 — `encode` / `decode`

So far we've only *trained* the tokenizer (learned the merge list). Now we
use it: take brand-new text, and replay the learned merges — in the exact
order we learned them — to turn it into tokens.

**ELI5:** you already decided the gluing order during training (glue "es"
first, then "est", then "low", etc.). Now, for any new word, you just apply
those same glue steps in that same order and see what tiles you end up with.
Words you've never seen still work — they just get broken into whatever
familiar pieces are left over.

In [ ]:
%%writefile src/tokenizer.py
from collections import defaultdict


def get_pair_counts(word_freqs):
    pair_counts = defaultdict(int)
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pair_counts[pair] += freq
    return pair_counts


def merge_pair(pair, word_freqs):
    new_word_freqs = {}
    for word, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_word_freqs[tuple(new_word)] = freq
    return new_word_freqs


def train_bpe(word_freqs, num_merges):
    word_freqs = dict(word_freqs)
    merges = []
    for step in range(num_merges):
        pair_counts = get_pair_counts(word_freqs)
        if not pair_counts:
            break
        best_pair = max(pair_counts, key=pair_counts.get)
        word_freqs = merge_pair(best_pair, word_freqs)
        merges.append(best_pair)
        print(f"Merge {step + 1}: {best_pair}  (count={pair_counts[best_pair]})")
    return word_freqs, merges


def get_word_tokens(word, merges):
    """Apply a learned merge list, in order, to a single word."""
    tokens = list(word) + ['</w>']
    for pair in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(tokens[i] + tokens[i + 1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


def encode(text, merges):
    """Whitespace-split text into words, then BPE-tokenize each word."""
    all_tokens = []
    for word in text.strip().split():
        all_tokens.extend(get_word_tokens(word, merges))
    return all_tokens


def decode(tokens):
    """Join tokens back into text."""
    text = ''.join(tokens).replace('</w>', ' ')
    return text.strip()

In [ ]:
import importlib
import tokenizer
importlib.reload(tokenizer)
from tokenizer import get_pair_counts, merge_pair, train_bpe, encode, decode

### Try it on a word the tokenizer never saw

`"lowest"` never appeared in training — only `"low"`, `"lower"`, `"newest"`,
`"widest"` did. Watch what happens.

In [ ]:
tokens = encode("zebra", merges)
tokens

In [ ]:
decode(tokens)

### Try it yourself!

As as expected, it shows up as
>"low", "est</w>"

Try with outher words:
- highest (not part of the training, so it should only show h,i,g,h,est)
- test (t,est)
- or something outrageous like zebra (z,e,b,r,a)

### Save your progress

Colab's runtime is ephemeral — anything not pushed disappears when it resets.

In [ ]:
!git add -A
!git commit -m "Module 0: add train_bpe and tokenizer walkthrough"
!git push